In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')





In [3]:
from google.colab import drive

drive.mount('/content/drive')
# Load your data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:

df = pd.read_excel('/content/drive/MyDrive/foodnutri.xlsx')
# Display first few rows
print("Food Data Overview:")
print(df[['Food Name', 'Energy (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fibre (g)']].head())



Food Data Overview:
          Food Name  Energy (kcal)  Carbohydrate (g)  Protein (g)  Fibre (g)
0    Rice and beans            165              28.4          7.2        3.6
1        White rice            130              28.2          2.6        0.4
2   White spaghetti            157              30.8          5.8        1.8
3       Jollof rice            158              30.2          3.8        0.8
4  Jollof spaghetti            172              31.4          6.2        1.6


In [ ]:
# Step 1: Define Nutritional Scoring Logic

In [6]:
def calculate_nutrition_score(row, bmi_category, goal):
    """
    Calculate a suitability score (0-100) based on BMI and Goal
    Higher score = more suitable for that profile
    """
    energy = row['Energy (kcal)']
    carbs = row['Carbohydrate (g)']
    protein = row['Protein (g)']
    fibre = row['Fibre (g)']

    score = 50  # Base score

    # === BMI-Based Adjustments ===
    if bmi_category == 'Underweight':
        # Need calorie-dense, nutrient-rich foods
        score += (energy / 50) * 2
        score += protein * 1.5
        score += fibre * 0.5
        if energy > 300:
            score += 10  # Bonus for high-calorie foods

    elif bmi_category == 'Overweight':
        # Need lower calories, higher protein/fibre for satiety
        score += protein * 2.5
        score += fibre * 2
        score -= (energy / 30)  # Penalize high calories
        if energy < 150:
            score += 10

    elif bmi_category == 'Obese':
        # Strong emphasis on low calorie, high protein/fibre
        score += protein * 3
        score += fibre * 2.5
        score -= (energy / 20)  # Heavy penalty for high calories
        if energy < 120:
            score += 15
        if fibre > 3:
            score += 5

    elif bmi_category == 'Normal_weight':
        # Balanced approach
        score += protein * 1.5
        score += fibre * 1
        score += (energy / 100)

    # === Goal-Based Adjustments ===
    if goal == 'Lose_weight':
        # Penalty for high calories, reward for protein and fibre
        score -= (energy / 25)
        score += protein * 3
        score += fibre * 2.5
        if carbs > 30 and energy > 200:
            score -= 10  # Penalty for high carb high calorie foods

    elif goal == 'Gain_Muscle':
        # Strong protein emphasis, moderate calories
        score += protein * 4
        score += (energy / 150)
        if protein > 20:
            score += 10  # Bonus for protein-rich foods
        if carbs > 40:
            score -= 5  # Slight penalty for very high carb

    elif goal == 'Gain_weight':
        # Encourage calorie-dense foods
        score += energy / 40
        score += protein * 1.5
        score += carbs * 0.5
        if energy > 300:
            score += 15  # Bonus for high-calorie foods

    elif goal == 'Maintain':
        # Moderate, balanced approach
        score += protein * 1.2
        score += fibre * 1
        score += energy / 200
        if 150 <= energy <= 250:
            score += 5  # Bonus for moderate calorie foods

    # Normalize to 0-100
    return np.clip(score, 0, 100)

In [7]:
# Step 2: Create Training Dataset

In [9]:
# Define all combinations of BMI and Goal
bmi_categories = ['Underweight', 'Normal_weight', 'Overweight', 'Obese']
goals = ['Lose_weight', 'Maintain', 'Gain_Muscle', 'Gain_weight']

# Create training data with all combinations
training_data = []

for bmi in bmi_categories:
    for goal in goals:
        temp_df = df.copy()
        temp_df['BMI_Category'] = bmi
        temp_df['Goal'] = goal
        temp_df['Score'] = temp_df.apply(
            lambda row: calculate_nutrition_score(row, bmi, goal), axis=1
        )
        training_data.append(temp_df)

training_df = pd.concat(training_data, ignore_index=True)

print(f"\nTraining dataset created with {len(training_df)} rows")
print("\nSample scores for different profiles:")
sample_foods = ['Rice and beans', 'Beef', 'Eba', 'Efo']
for food in sample_foods:
    print(f"\n{food}:")
    for bmi in ['Normal_weight', 'Obese']:
        for goal in ['Lose_weight', 'Gain_Muscle']:
            score = training_df[
                (training_df['Food Name'] == food) &
                (training_df['BMI_Category'] == bmi) &
                (training_df['Goal'] == goal)
            ]['Score'].values[0]
            print(f"  {bmi} + {goal}: {score:.1f}/100")
print("\nTraining dataset created with rows")
print(training_df.head())


Training dataset created with 608 rows

Sample scores for different profiles:

Rice and beans:
  Normal_weight + Lose_weight: 90.0/100
  Normal_weight + Gain_Muscle: 95.9/100
  Obese + Lose_weight: 100.0/100
  Obese + Gain_Muscle: 100.0/100

Beef:
  Normal_weight + Lose_weight: 100.0/100
  Normal_weight + Gain_Muscle: 100.0/100
  Obese + Lose_weight: 100.0/100
  Obese + Gain_Muscle: 100.0/100

Eba:
  Normal_weight + Lose_weight: 38.5/100
  Normal_weight + Gain_Muscle: 58.6/100
  Obese + Lose_weight: 20.6/100
  Obese + Gain_Muscle: 40.7/100

Efo:
  Normal_weight + Lose_weight: 86.5/100
  Normal_weight + Gain_Muscle: 88.8/100
  Obese + Lose_weight: 100.0/100
  Obese + Gain_Muscle: 100.0/100

Training dataset created with rows
   S/N         Food Name NFCT Code Category  Energy (kcal)  Carbohydrate (g)  \
0    1    Rice and beans   NG-0201    Grain            165              28.4   
1    2        White rice   NG-0202    Grain            130              28.2   
2    3   White spaghetti 

In [10]:
# Step 3: Train Random Forest Model

In [11]:
# Encode categorical variables
le_bmi = LabelEncoder()
le_goal = LabelEncoder()

training_df['BMI_Encoded'] = le_bmi.fit_transform(training_df['BMI_Category'])
training_df['Goal_Encoded'] = le_goal.fit_transform(training_df['Goal'])

# Prepare features
features = ['Energy (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fibre (g)',
            'BMI_Encoded', 'Goal_Encoded']

X = training_df[features]
y = training_df['Score']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale numeric features
scaler = StandardScaler()
numeric_cols = ['Energy (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fibre (g)']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_scaled)
print(f"\nModel Performance:")
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)


Model Performance:
R² Score: 0.853
RMSE: 5.64

Feature Importance:
            Feature  Importance
2       Protein (g)    0.669620
5      Goal_Encoded    0.139224
1  Carbohydrate (g)    0.078449
4       BMI_Encoded    0.054654
0     Energy (kcal)    0.036502
3         Fibre (g)    0.021551


In [12]:
# Step 4: Scoring Function

In [13]:
def score_foods_for_profile(df, bmi_category, goal, model, scaler, le_bmi, le_goal):
    """
    Score all foods for a specific BMI category and goal
    """
    # Encode the profile
    bmi_encoded = le_bmi.transform([bmi_category])[0]
    goal_encoded = le_goal.transform([goal])[0]

    scores = []
    for idx, row in df.iterrows():
        # Prepare input features
        input_data = {
            'Energy (kcal)': row['Energy (kcal)'],
            'Carbohydrate (g)': row['Carbohydrate (g)'],
            'Protein (g)': row['Protein (g)'],
            'Fibre (g)': row['Fibre (g)'],
            'BMI_Encoded': bmi_encoded,
            'Goal_Encoded': goal_encoded
        }

        input_df = pd.DataFrame([input_data])

        # Scale numeric features
        numeric_cols = ['Energy (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fibre (g)']
        input_df[numeric_cols] = scaler.transform(input_df[numeric_cols])

        # Predict score
        score = model.predict(input_df)[0]
        scores.append(np.clip(score, 0, 100))

    # Create results dataframe
    result_df = df.copy()
    result_df['Score'] = scores
    result_df['BMI'] = bmi_category
    result_df['Goal'] = goal

    # Add classification
    result_df['Recommendation'] = pd.cut(
        result_df['Score'],
        bins=[0, 30, 50, 70, 100],
        labels=['Avoid', 'Fair', 'Good', 'Excellent']
    )

    return result_df.sort_values('Score', ascending=False)

In [14]:
# Step 5: Get Recommendations for Any Profile

In [15]:
# Example 1: Obese person trying to lose weight
print("\n" + "="*80)
print("RECOMMENDATIONS FOR OBESE + LOSE_WEIGHT")
print("="*80)

profile = 'Obese'
goal = 'Lose_weight'

results = score_foods_for_profile(
    df, profile, goal, rf_model, scaler, le_bmi, le_goal
)

# Display top recommendations
display_cols = ['Food Name', 'Category', 'Energy (kcal)', 'Protein (g)',
                'Fibre (g)', 'Score', 'Recommendation']

print(f"\nTop 10 Foods for {profile} + {goal}:")
print(results[display_cols].head(10).to_string(index=False))

print(f"\nBottom 10 Foods for {profile} + {goal} (to avoid):")
print(results[display_cols].tail(10).to_string(index=False))


RECOMMENDATIONS FOR OBESE + LOSE_WEIGHT

Top 10 Foods for Obese + Lose_weight:
  Food Name Category  Energy (kcal)  Protein (g)  Fibre (g)      Score Recommendation
  Goat meat  Protein            204         27.2        0.0 100.000000      Excellent
     Turkey  Protein            189         28.6        0.0 100.000000      Excellent
Ringed fish  Protein            338         62.4        0.0 100.000000      Excellent
   Cut fish  Protein            148         22.6        0.0 100.000000      Excellent
       Beef  Protein            218         26.4        0.0 100.000000      Excellent
      Ponmo  Protein            163         36.8        0.0 100.000000      Excellent
    Chicken  Protein            215         27.8        0.0 100.000000      Excellent
        Egg  Protein            155         12.6        0.0  99.817431      Excellent
    Gbegiri     Soup            118          8.4        4.8  99.770208      Excellent
    Moi moi   Legume            130          9.4        3.4 

In [16]:
# Step 6: Compare Different Profiles

In [17]:
# Compare scores for a specific food across different profiles
def compare_food_scores(df, food_name):
    """
    Show how a specific food scores across all BMI+Goal combinations
    """
    results = []
    for bmi in bmi_categories:
        for goal in goals:
            scored = score_foods_for_profile(df, bmi, goal, rf_model, scaler, le_bmi, le_goal)
            food_score = scored[scored['Food Name'] == food_name]['Score'].values[0]
            results.append({
                'BMI': bmi,
                'Goal': goal,
                'Score': food_score
            })

    return pd.DataFrame(results).pivot(index='BMI', columns='Goal', values='Score')

# Example: Compare "Eba" across profiles
print("\n" + "="*80)
print("SCORES FOR 'EBA' ACROSS ALL PROFILES")
print("="*80)
eba_scores = compare_food_scores(df, 'Eba')
print(eba_scores.round(1))


SCORES FOR 'EBA' ACROSS ALL PROFILES
Goal           Gain_Muscle  Gain_weight  Lose_weight  Maintain
BMI                                                           
Normal_weight         53.3         95.5         39.1      50.9
Obese                 51.1         94.6         36.2      47.5
Overweight            52.6         95.6         38.7      49.4
Underweight           55.8         96.8         61.7      68.0


In [18]:
# Step 7: Comprehensive Analysis Function

In [19]:
def analyze_all_profiles(df):
    """
    Generate complete scoring for all BMI + Goal combinations
    """
    all_results = {}

    for bmi in bmi_categories:
        for goal in goals:
            key = f"{bmi}_{goal}"
            results = score_foods_for_profile(df, bmi, goal, rf_model, scaler, le_bmi, le_goal)
            all_results[key] = results[['Food Name', 'Score']]

    return all_results

# Get all profiles
all_scores = analyze_all_profiles(df)

# Example: Show top 5 foods for each profile
print("\n" + "="*80)
print("TOP 5 FOODS FOR EACH BMI + GOAL COMBINATION")
print("="*80)

for profile, scores_df in all_scores.items():
    bmi, goal = profile.split('_')
    print(f"\n{bmi} + {goal}:")
    print(scores_df.head(5).to_string(index=False))


TOP 5 FOODS FOR EACH BMI + GOAL COMBINATION


ValueError: too many values to unpack (expected 2)

In [20]:
def analyze_all_profiles(df):
    """
    Generate complete scoring for all BMI + Goal combinations
    """
    all_results = {}

    for bmi in bmi_categories:
        for goal in goals:
            key = f"{bmi}|{goal}"  # Using | instead of _ to avoid split issues
            results = score_foods_for_profile(df, bmi, goal, rf_model, scaler, le_bmi, le_goal)
            all_results[key] = results[['Food Name', 'Score']]

    return all_results

# Get all profiles
all_scores = analyze_all_profiles(df)

# Example: Show top 5 foods for each profile
print("\n" + "="*80)
print("TOP 5 FOODS FOR EACH BMI + GOAL COMBINATION")
print("="*80)

for profile, scores_df in all_scores.items():
    bmi, goal = profile.split('|')  # Split on | instead of _
    print(f"\n{bmi} + {goal}:")
    print(scores_df.head(5).to_string(index=False))


TOP 5 FOODS FOR EACH BMI + GOAL COMBINATION

Underweight + Lose_weight:
  Food Name  Score
  Goat meat  100.0
     Turkey  100.0
Ringed fish  100.0
   Cut fish  100.0
       Beef  100.0

Underweight + Maintain:
  Food Name  Score
  Goat meat  100.0
     Turkey  100.0
    Chicken  100.0
Ringed fish  100.0
       Beef  100.0

Underweight + Gain_Muscle:
  Food Name  Score
  Goat meat  100.0
     Turkey  100.0
Ringed fish  100.0
   Cut fish  100.0
       Beef  100.0

Underweight + Gain_weight:
  Food Name  Score
  Goat meat  100.0
     Turkey  100.0
Ringed fish  100.0
   Cut fish  100.0
       Beef  100.0

Normal_weight + Lose_weight:
Food Name  Score
Goat meat  100.0
   Turkey  100.0
  Chicken  100.0
 Cut fish  100.0
     Beef  100.0

Normal_weight + Maintain:
  Food Name  Score
  Goat meat  100.0
     Turkey  100.0
    Chicken  100.0
Ringed fish  100.0
       Beef  100.0

Normal_weight + Gain_Muscle:
Food Name  Score
Goat meat  100.0
   Turkey  100.0
  Chicken  100.0
 Cut fish  100.0
  

In [ ]:
# Step 8: Export Results

In [21]:
# Export scores for a specific profile to Excel
def export_profile_scores(df, bmi_category, goal):
    """
    Export scored foods for a specific profile to Excel
    """
    results = score_foods_for_profile(df, bmi_category, goal, rf_model, scaler, le_bmi, le_goal)

    # Add additional metrics
    results['Energy_Per_Score'] = results['Energy (kcal)'] / results['Score']
    results['Protein_Per_Score'] = results['Protein (g)'] / results['Score']

    # Save to Excel
    filename = f'food_scores_{bmi_category}_{goal}.xlsx'
    results.to_excel(filename, index=False)
    print(f"\nResults exported to {filename}")

    return results

# Example: Export for Obese + Lose Weight
export_profile_scores(df, 'Obese', 'Lose_weight')


Results exported to food_scores_Obese_Lose_weight.xlsx


,S/N,Food Name,NFCT Code,Category,Energy (kcal),Carbohydrate (g),Protein (g),Fibre (g),Calorie Classification,Carbohydrate Classification,Protein Classification,Fibre Classification,Score,BMI,Goal,Recommendation,Energy_Per_Score,Protein_Per_Score
36,37,Goat meat,NG-0304,Protein,204,0.0,27.2,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,2.040000,0.272000
29,30,Turkey,NG-0303,Protein,189,0.0,28.6,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,1.890000,0.286000
17,18,Ringed fish,NG-0308,Protein,338,0.0,62.4,0.0,High calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,3.380000,0.624000
18,19,Cut fish,NG-0309,Protein,148,0.0,22.6,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,1.480000,0.226000
13,14,Beef,NG-0301,Protein,218,0.0,26.4,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,2.180000,0.264000
12,13,Ponmo,NG-0307,Protein,163,0.0,36.8,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,1.630000,0.368000
23,24,Chicken,NG-0302,Protein,215,0.0,27.8,0.0,Moderate calorie,Low carbohydrate,Protein rich,Low fibre,100.000000,Obese,Lose_weight,Excellent,2.150000,0.278000
14,15,Egg,NG-0305,Protein,155,1.1,12.6,0.0,Moderate calorie,Low carbohydrate,Moderate protein,Low fibre,99.817431,Obese,Lose_weight,Excellent,1.552835,0.126230
26,27,Gbegiri,NG-0244,Soup,118,14.6,8.4,4.8,Low calorie,Low carbohydrate,Moderate protein,Source of fibre,99.770208,Obese,Lose_weight,Excellent,1.182718,0.084193
20,21,Moi moi,NG-0178,Legume,130,14.2,9.4,3.4,Moderate calorie,Low carbohydrate,Moderate protein,Source of fibre,99.671283,Obese,Lose_weight,Excellent,1.304287,0.094310


In [ ]:
# Complete Interactive Function

In [22]:
def get_food_recommendations(bmi, goal, top_n=10):
    """
    Main function to get food recommendations for any profile
    """
    # Validate inputs
    valid_bmi = ['Underweight', 'Normal_weight', 'Overweight', 'Obese']
    valid_goals = ['Lose_weight', 'Maintain', 'Gain_Muscle', 'Gain_weight']

    if bmi not in valid_bmi:
        raise ValueError(f"BMI must be one of {valid_bmi}")
    if goal not in valid_goals:
        raise ValueError(f"Goal must be one of {valid_goals}")

    # Get scores
    results = score_foods_for_profile(df, bmi, goal, rf_model, scaler, le_bmi, le_goal)

    # Summary statistics
    print(f"\n{'='*60}")
    print(f"FOOD RECOMMENDATIONS FOR: {bmi} + {goal}")
    print(f"{'='*60}")
    print(f"Total foods scored: {len(results)}")
    print(f"Average score: {results['Score'].mean():.1f}/100")
    print(f"Highest score: {results['Score'].max():.1f}/100")
    print(f"Lowest score: {results['Score'].min():.1f}/100")

    # Top recommendations
    display_cols = ['Food Name', 'Category', 'Energy (kcal)', 'Protein (g)',
                    'Fibre (g)', 'Score', 'Recommendation']

    print(f"\nTop {top_n} Recommended Foods:")
    print(results[display_cols].head(top_n).to_string(index=False))

    return results

# Example usage:
# recommendations = get_food_recommendations('Overweight', 'Gain_Muscle', top_n=10)